# 09 — Time Score

## Goal
Score how **many candles** make up the base of a zone.  
Fewer candles = the market made a faster decision at that level = stronger institutional conviction.

---

## Rationale (sd-concepts.md §8)

A single explosive candle that reverses immediately shows the clearest institutional footprint.  
A 4–5 candle base shows indecision: too much back-and-forth before the breakout.

| Base candle count | Time score | Meaning |
|-------------------|------------|---------|
| 1–2               | 2          | Explosive — highest conviction |
| 3                 | 1          | Compact — acceptable |
| ≥ 4               | 0          | Indecisive — weakened conviction |

---

## Equation

$$\text{time\_score} =
\begin{cases}
2 & n \leq 2 \\
1 & n = 3 \\
0 & n \geq 4
\end{cases}
\qquad n = \text{end} - \text{start} + 1$$

---

**Note:** `time_score` is one of five independent additive inputs to the final **S.E.T.S** total (§10).  
It does not combine with freshness here — each score stays separate.

## 1. Setup & load data

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import plotly.graph_objects as go

from utils.data_loader import load_all_timeframes
from utils.models import CandlePrimitives, add_atr
from utils.config import (
    COLOR_BULL, COLOR_BEAR,
    CHART_BG as BG, CHART_GRID as GRID,
    TIME_SCORE_TABLE,
)
from utils.base_detector import detect_bases
from utils.legs_formation import detect_formations
from utils.zone_detector import detect_zones
from utils.freshness import add_freshness
from utils.time_scoring import add_time_score

SYMBOL = "USDJPY=X"

raw = load_all_timeframes(SYMBOL, align=False)
data = {tf: add_atr(CandlePrimitives.enrich_dataframe(df)) for tf, df in raw.items()}
common_start = max(df.index[0] for df in raw.values())
data = {tf: df[df.index >= common_start] for tf, df in data.items()}

print("Loaded timeframes:", list(data.keys()))
for tf, df in data.items():
    print(f"  {tf}: {len(df)} candles  ({df.index[0].date()} → {df.index[-1].date()})")

Loaded timeframes: ['1wk', '1d', '4h', '1h']
  1wk: 104 candles  (2024-06-09 → 2026-05-31)
  1d: 581 candles  (2024-06-06 → 2026-06-05)
  4h: 3174 candles  (2024-06-06 → 2026-06-05)
  1h: 12259 candles  (2024-06-05 → 2026-06-05)


## 2. Run the full pipeline with all scores

In [2]:
results = {}

for tf, df in data.items():
    passed_bases, _  = detect_bases(df)
    formations       = detect_formations(df, passed_bases)
    passed, rejected = detect_zones(df, formations)

    for zones in (passed, rejected):
        add_freshness(df, zones)
        add_time_score(zones)

    results[tf] = dict(df=df, passed_zones=passed, rejected=rejected)

# Summary table
print(f"{'TF':<6} {'Zones':>6} {'t=2':>5} {'t=1':>5} {'t=0':>5}")
print("-" * 30)
for tf, r in results.items():
    p = r["passed_zones"]
    print(
        f"{tf:<6} {len(p):>6}"
        f"  {sum(1 for z in p if z['time_score']==2):>3}"
        f"  {sum(1 for z in p if z['time_score']==1):>3}"
        f"  {sum(1 for z in p if z['time_score']==0):>3}"
    )

TF      Zones   t=2   t=1   t=0
------------------------------
1wk         2    2    0    0
1d         24   23    1    0
4h        133  122    5    6
1h        421  358   36   27


## 3. Detail table — passed zones

In [3]:
for tf, r in results.items():
    p = r["passed_zones"]
    if not p:
        print(f"\n{tf}: no passed zones")
        continue

    rows = []
    for z in p:
        rows.append({
            "formation":       z["formation"],
            "zone_type":       z["zone_type"],
            "proximal":        round(z["proximal"], 3),
            "base_count":      z["base_count"],
            "time_score":      z["time_score"],
            "freshness_score": z["freshness_score"],
        })

    print(f"\n{'═'*55}  {tf}")
    print(pd.DataFrame(rows).to_string(index=False))


═══════════════════════════════════════════════════════  1wk
formation zone_type  proximal  base_count  time_score  freshness_score
      DBD    supply   155.497           1           2                0
      DBR    demand   151.217           1           2                0

═══════════════════════════════════════════════════════  1d
formation zone_type  proximal  base_count  time_score  freshness_score
      RBD    supply   156.280           1           2                0
      RBD    supply   152.405           2           2                0
      DBD    supply   148.506           1           2                0
      RBD    supply   145.768           2           2                0
      DBR    demand   149.626           1           2                0
      DBD    supply   153.545           1           2                0
      DBD    supply   150.976           1           2                0
      RBR    demand   154.340           1           2                0
      DBR    demand   148

## 4. Visualize — time score chart for all 4 timeframes

Zone border colour reflects the time score:
- **Green** `#26a69a` → score 2 (explosive)
- **Orange** `#ffb74d` → score 1 (compact)
- **Red** `#ef5350` → score 0 (indecisive)

In [ ]:
_TIME_COLOR = {2: "#26a69a", 1: "#ffb74d", 0: "#ef5350"}
_EDGE_COLOR = {"demand": "#26a69a", "supply": "#ef5350"}


def plot_time_score(tf: str, symbol: str, r: dict, plot_months: int = 36) -> None:
    """Candlestick chart with zone borders colour-coded by time score."""
    df        = r["df"]
    all_zones = r["passed_zones"] + r["rejected"]

    cutoff  = df.index[-1] - pd.DateOffset(months=plot_months)
    df_plot = df[df.index >= cutoff].copy()
    offset  = len(df) - len(df_plot)

    vis_zones    = [z for z in all_zones if z["end"] >= offset]
    n_passed_all = len(r["passed_zones"])
    n_rej_all    = len(r["rejected"])

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=df_plot.index,
        open=df_plot["open"], high=df_plot["high"],
        low=df_plot["low"],   close=df_plot["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ))

    for z in vis_zones:
        bs_plot = max(z["start"] - offset, 0)
        be_plot = min(z["end"]   - offset, len(df_plot) - 1)
        passed  = z["passed"]
        tscore  = z.get("time_score", 0)
        zt      = z["zone_type"]

        x0 = df_plot.index[bs_plot]
        x1 = df_plot.index[be_plot]
        border_col = _TIME_COLOR[tscore] if passed else "#b0bec5"

        fig.add_vrect(
            x0=x0, x1=x1,
            fillcolor="rgba(124,131,253,0.10)" if passed else "rgba(176,190,197,0.03)",
            line_color=border_col,
            line_width=1,
            line_dash="solid" if passed else "dot",
            opacity=1.0,
        )

        ext_iloc = min(z["end"] - offset + 4, len(df_plot) - 1)
        x_ext = df_plot.index[ext_iloc]
        fig.add_shape(
            type="line", x0=x0, x1=x_ext,
            y0=z["proximal"], y1=z["proximal"],
            line=dict(color=_EDGE_COLOR[zt], width=1, dash="dash"),
        )
        fig.add_shape(
            type="line", x0=x0, x1=x_ext,
            y0=z["distal"], y1=z["distal"],
            line=dict(color=_EDGE_COLOR[zt], width=1, dash="dot"),
        )

        if passed:
            bc    = z.get("base_count", "?")
            label = f"{z['formation']} ✓<br>t={tscore} ({bc}c)"
        else:
            label = f"{z['formation']} ✗"

        fig.add_annotation(
            x=x0, y=z["proximal"],
            text=label,
            showarrow=False,
            font=dict(size=8, color=border_col),
            xanchor="left", yanchor="bottom",
        )

    fig.update_layout(
        title=(
            f"{symbol} {tf} — Time Score  (last {plot_months} months)  "
            f"full history: {n_passed_all} passed, {n_rej_all} rejected"
        ),
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        xaxis=dict(gridcolor=GRID),
        yaxis=dict(gridcolor=GRID, title="Price"),
        height=500,
    )
    fig.show()


for tf, r in results.items():
    plot_time_score(tf=tf, symbol=SYMBOL, r=r, plot_months=36)

## 5. What's next

**NB10 — Curve Position**

Not all zones on a chart are equally relevant to current price.  
A demand zone sitting 15% below current price is unlikely to be reached anytime soon.  
The next notebook scores each zone by its **position relative to the current price curve** —  
zones close to current price get a higher positional score,  
contributing to the final composite ranking alongside freshness and time.